# Floorplan Vectorization: a Classical Computer-Vision Pipeline

**Signal, Image and Video Processing, course project**

## Goal

Convert a raster floorplan image (PNG/JPG technical drawing) into **vector wall geometry**: a clean set of straight line segments describing the building's wall network, using only classical image-processing techniques.
The resulting svg is intended to be used in applications that do not require a perfect "translation" of the original image.

## Pipeline overview

The notebook is organized as a **linear pipeline**, where each stage consumes the output of the previous one. Every stage is a standalone, independently testable function:

| # | Stage | Function(s) | Input -> Output |
|---|-------|-------------|-----------------|
| 1 | Preprocessing | `preprocess` | Raw BGR image -> inverse binary mask |
| 2 | Wall isolation | `isolate_walls_multiscale` | Binary mask -> wall-only mask |
| 3 | Line detection | `detect_walls` (Hough) / `detect_walls_lsd` (LSD) | Wall mask -> raw `[x1,y1,x2,y2]` segments |
| 4 | Regularization | `regularize` (`snap_segments_angle` -> `bridge_wall_gaps` -> `snap_corners`) | Raw segments -> cleaner segments |
| 5 | Export | `segments_to_svg` | Segments -> a real `.svg` vector file |
| 6 | Evaluation | `verify_single` / `verify_dataset` against CVC-FP ground truth | Segments + GT SVG -> precision / recall / F1 |
| 7 | Performance tracking | `log_performance` / `plot_history` | Evaluation results -> graphs |


Below: the pipeline end to end, from raw scan to CAD-ready export. Solid arrows are the main scored path (stages 1-7); the CAD-ready branch off of regularization is the ad hoc, single-image path described later, not wired into evaluation.

![pipeline architecture](NB%20IMAGES/00_pipeline_architecture.png)

### Dependencies

- `cv2`: image and video processing functions.
- `os`: filesystem interaction.
- `numpy`: array manipulation and vectorized geometry.
- `matplotlib.pyplot`: for visualization.
- `skimage.morphology.skeletonize`: topological thinning.
- `sklearn.cluster.DBSCAN`: density-based clustering.

In [41]:
import cv2
import os
import matplotlib.pyplot as plt
import numpy as np
from skimage.morphology import skeletonize
from sklearn.cluster import DBSCAN

In [42]:
def select_images(path): # can pass a single image or a folder
    """
    Collect image paths from a single file or a directory.

    Args:
        path: Path to an image file or a folder of images.

    Returns:
        Paths to the supported images found (.png/.jpg/.jpeg).
    """
    supported_formats = (".png", ".jpg", ".jpeg")
    images = []

    # single image
    if os.path.isfile(path):
        if path.lower().endswith(supported_formats):
            images.append(path)
        else:
            print(f"File {path} has not a supported image format")

    # folder
    elif os.path.isdir(path):       
        for filename in os.listdir(path):
            if filename.lower().endswith(supported_formats):
                img_path = os.path.join(path, filename)
                images.append(img_path)

    return images # list of images to process

In [43]:
def show_images(images: dict):
    """
    Display one or more images side by side for visual inspection.

    Args:
        images: Mapping of subplot title -> image.

    Returns:
        None.
    """

    # get size from first image
    first = list(images.values())[0]
    h, w = first.shape[:2]
    
    # subplots because using plot it gets overwritten
    # figsize proportional to img resolution, to avoid distortion
    _, axes = plt.subplots(1, len(images), figsize=(w/100*len(images), h/100))
    
    for ax, (title, img) in zip(axes, images.items()):
        if len(img.shape) == 3:
            ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        else:
            ax.imshow(img, cmap='gray')
        ax.set_title(title)
        ax.axis('off')
    
    plt.tight_layout()
    plt.show()

## Preprocessing

### `preprocess` - grayscale, denoise, binarize

Three classical steps, in order:

1. **Grayscale conversion**: floorplan drawings carry no color information relevant to wall geometry, so a single intensity channel is sufficient and cheaper for everything downstream.
2. **Gaussian blur**: suppresses scanning noise and JPEG artifacts so binarization doesn't pick up isolated pixels as foreground.
3. **Inverse Otsu thresholding** : Otsu's method automatically picks the threshold that best separates the intensity histogram into two classes (white paper background vs dark ink), which works well since floorplan scans are close to bimodal. The inverse variant puts walls/lines white on a black background, since every later stages expect foreground-as-white.

![preprocessing steps](NB%20IMAGES/01_preprocessing.png)


In [44]:
def preprocess(img, visualize=False):
    """
    Convert a raw image into a binary, white-on-black wall mask.

    Args:
        img: Original BGR image.

    Returns:
        Binary image.
    """
    gray_img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    noise_reduced_img = cv2.GaussianBlur(gray_img, (5, 5), 0) # 0 to let it compute sigma
    _, binary_img = cv2.threshold(noise_reduced_img, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)

    if visualize:
        show_images({
            "Original": img,
            "Grayscale": gray_img,
            "Blurred": noise_reduced_img,
            "Inverse Binary": binary_img
        })

    return binary_img



### `isolate_walls` - single-scale wall isolation

`cv2.distanceTransform` replaces every foreground (white) pixel with its Euclidean distance to the nearest background (black) pixel. Thresholding this map (`> 2.0`) keeps only pixels that are part of a sufficiently thick stroke, i.e. walls, and discards thin ink. A `MORPH_CLOSE` (5x5 kernel) then patches small gaps left in the wall mask.

In [45]:
def isolate_walls(img, visualize=False):
    """
    Isolate wall regions from a binary image using a single distance threshold.

    Args:
        img: Binary image from preprocess().

    Returns:
        Binary mask of wall regions.
    """
    
    # each pixel is substituted by its distance to the nearest black pixel
    dist_img = cv2.distanceTransform(img, cv2.DIST_L2, 5)
    
    # keep only thick lines (= walls)
    _, walls_img = cv2.threshold(dist_img, 2.0, 255, cv2.THRESH_BINARY)

    # close small gaps in walls
    kernel = np.ones((5, 5), np.uint8)
    walls_img = cv2.morphologyEx(walls_img, cv2.MORPH_CLOSE, kernel)

    if visualize:
        show_images({
            "Binary": img,
            "Distance map": dist_img,
            "Walls only": walls_img
        })

    return walls_img

**Below:** `isolate_walls` on a sample floorplan. The distance transform (middle) is brightest along each stroke's centerline and fades to black near its edges; thresholding it at `> 2.0` keeps only pixels deep enough inside a stroke to belong to a wall rather than thin ink (room labels, dimension lines, furniture symbols), which is why those disappear in the right panel while wall strokes survive, slightly thinned.

![isolate_walls](NB%20IMAGES/02a_isolate_walls.png)

### `isolate_walls_multiscale` - two-band wall isolation

This function is intended to be used in future extensions or pipeline improvents, to differentiate thick and thin walls behaviour and analysis. It extends the single-scale idea with two thresholds on the same distance-transform map, applied to disjoint distance ranges:

- **`thick`**: `distance > thick_dist` (default 7.0), pixels deep inside a thick stroke, i.e. structural/exterior walls.
- **`thin`**: `thin_dist < distance <= thick_dist` (default `2.0 < d <= 7.0`), obtained as `threshold(dist, thin_dist) - thick`. This band captures thinner interior partition walls, and unfortunately some non-wall thin strokes too.
- **`all_walls`**: `cv2.bitwise_or(thick, thin)`, the two bands recombined into one wall mask.

![wall isolation steps](NB%20IMAGES/02_wall_isolation.png)

In [46]:
def isolate_walls_multiscale(binary_img, thick_dist=7.0, thin_dist=2.0, visualize=False):
    """
    Isolate wall regions at multiple thickness scales and recombine them.

    Args:
        binary_img: Binary image from preprocess().
        thick_dist: Distance-transform threshold for the thick (structural) wall band.
        thin_dist: Distance-transform threshold for the thin (partition) wall band.

    Returns:
        Binary mask of all walls.
    """
    dist = cv2.distanceTransform(binary_img, cv2.DIST_L2, 5)

    # thick walls
    _, thick = cv2.threshold(dist, thick_dist, 255, cv2.THRESH_BINARY)
    thick = thick.astype(np.uint8)

    # thin walls
    _, thin = cv2.threshold(dist, thin_dist, 255, cv2.THRESH_BINARY)
    thin = thin.astype(np.uint8)
    thin = cv2.subtract(thin, thick) # remove thick walls already detected

    # all walls
    all_walls = cv2.bitwise_or(thick, thin)

    if visualize:
        show_images({
            "Binary":   binary_img,
            "Thick":    thick,
            "Thin":     thin,
            "Combined": all_walls
        })

    return all_walls


## Line detection

### `detect_walls` - skeletonize + probabilistic Hough transform

Two steps:

1. **Skeletonization**: thins every wall stroke down to a topologically-connected 1-pixel-wide centerline. Run directly on a wall mask,  skeletonizing first is crucial for HoughLinesP application since it collapses each wall to a single centerline and so Hough returns one line per wall.
2. **Probabilistic Hough transform** on the skeleton: `rho=1` (1-pixel radial resolution), `theta=pi/180` (1-degree angular resolution), and `threshold`/`minLineLength`/`maxLineGap` computed as `min(h, w) * fraction` (1%, 1%, 0.5% respectively) rather than fixed pixel counts, resolution-independent so the function behaves comparably for every possible img size.

The raw `cv2.HoughLinesP` output `[[[x1,y1,x2,y2]], [[x3,y3,x4,y4]], ...]` is flattened to a plain list of `[x1, y1, x2, y2]`.

![Hough line detection](NB%20IMAGES/03_line_detection_hough.png)


In [47]:
def detect_walls(img, threshold_frac=0.02, minlen_frac=0.01, maxgap_frac=0.0025, rho_px=1, theta_deg=1, visualize=False):
    """
    Detect straight wall segments via skeletonization and the Hough transform.

    Args:
        img: Binary walls mask.
        threshold_frac: HoughLinesP threshold, as a fraction of min(h, w).
        minlen_frac: HoughLinesP minLineLength, as a fraction of min(h, w).
        maxgap_frac: HoughLinesP maxLineGap, as a fraction of min(h, w).
        rho_px: HoughLinesP rho, the accumulator distance resolution in pixels.
        theta_deg: HoughLinesP theta, the accumulator angle resolution in degrees.

    Returns:
        Detected segments as [x1, y1, x2, y2].
    """

    # thin img -> 1px lines
    skeletonized_img = skeletonize(img // 255).astype(np.uint8) * 255 # skeletonize expects a binary image with 0/1 values

    # to have hough transform parameters proportional to img size
    h, w = skeletonized_img.shape

    # detect img (straight segments)
    detected_segments = cv2.HoughLinesP(
        skeletonized_img,
        rho = rho_px,                               # rho buckets of rho_px pixels
        theta = np.deg2rad(theta_deg),               # angle buckets of theta_deg degrees
        threshold = int(min(h, w) * threshold_frac),
        minLineLength = int(min(h, w) * minlen_frac),
        maxLineGap = int(min(h, w) * maxgap_frac)
    )

    # HoughLinesP returns None (not an empty array) when it finds zero lines
    detected_segments = [] if detected_segments is None else [seg[0].tolist() for seg in detected_segments] # unwrap the output so to have a list of segments

    height, width = img.shape
    drawn_walls_img = np.zeros((height, width, 3), dtype=np.uint8)
    for segment in detected_segments:
        x1 = segment[0]
        y1 = segment[1]
        x2 = segment[2]
        y2 = segment[3]
        cv2.line(drawn_walls_img, (x1, y1), (x2, y2), (0, 255, 0), 2)

    if visualize:
        show_images({
            "Walls": img,
            "Skeleton": skeletonized_img,
            f"Detected segments ({len(detected_segments) if detected_segments is not None else 0})": drawn_walls_img
        })

    return detected_segments


### `detect_walls_lsd` - Line Segment Detector

`cv2.createLineSegmentDetector` implements the LSD algorithm (explained in "LSD: a Line Segment Detector" by Grompone von Gioi, Jakubowicz, Morel, and Randall): a linear-time, contour-based segment detector that estimates local gradient direction at every pixel and grows line-support regions directly, with no free parameters to tune (`0` selects the standard, non-scale-refined variant). It is run on the skeletonized wall mask, same as `detect_walls`, so that the two detectors are directly comparable on identical input rather than differing in both algorithm and preprocessing.

Output is converted to the same flat `[x1, y1, x2, y2]` format, so `detect_walls` and `detect_walls_lsd` are interchangeable.

**Below:** LSD on the same wall mask and skeleton as above, **79 segments** vs Hough's 63. Even on the identical 1px skeleton, LSD still fragments some walls into more, shorter pieces than Hough does.

![LSD line detection](NB%20IMAGES/03b_line_detection_lsd.png)


In [48]:
def detect_walls_lsd(walls, visualize=False):
    """
    Detect straight wall segments using OpenCV's Line Segment Detector.

    Args:
        walls: Binary walls mask.

    Returns:
        Detected segments as [x1, y1, x2, y2].
    """

    # run on skeletonized walls so that LSD and HOUGH are comparable
    skeleton = skeletonize(walls // 255).astype(np.uint8) * 255

    lsd = cv2.createLineSegmentDetector(0)
    lines, _, _, _ = lsd.detect(skeleton)
    
    # draw results
    canvas = np.zeros((*walls.shape, 3), dtype=np.uint8)
    lsd.drawSegments(canvas, lines)
    
    # convert to same flat format as before
    segments = []
    if lines is not None:
        for line in lines:
            x1, y1, x2, y2 = map(int, line[0])
            segments.append([x1, y1, x2, y2])
    
    if visualize:
        show_images({
            "Walls mask": walls,
            "Skeleton (1px)": skeleton,
            f"LSD segments (n={len(segments)})": canvas
        })

    return segments

## Regularization

### `snap_angle` / `snap_segments_angle`

Architectural floorplans are drawn almost exclusively with walls at 0/90 degrees (axis-aligned) or occasionally 45/135 degrees (diagonal walls). Raw Hough/LSD segments rarely land on these exact angles.

- `snap_angle(angle, tolerance=10)`: computes the closest of the five canonical angles `[0, 45, 90, 135, 180]` and returns it only if the original angle is within 10 degrees of it; otherwise the angle is returned unchanged, treated as a genuinely non-canonical orientation rather than forced to a wrong snap target.
- `snap_segments_angle(segments)`: for each segment, computes its orientation via `atan2(y2-y1, x2-x1)`, snaps it, and only if the angle actually changed, recomputes the second endpoint by rotating around the first endpoint while preserving the segment's original length.


In [49]:
def snap_angle(angle_degree, tolerance=5):
    """
    Snap an angle to the nearest target angle if within tolerance.

    Args:
        angle_degree: Angle in degrees to snap.
        tolerance: Maximum degrees of difference allowed to snap.

    Returns:
        The snapped angle, or the original angle if no target was close enough.
    """
    target_angles = [0, 45, 90, 135, 180]
    # compute the closest target angle to the given angle
    closest_angle = min(target_angles, key=lambda t: abs(angle_degree - t))

    if abs(angle_degree - closest_angle) <= tolerance:
        return closest_angle
    return angle_degree

def snap_segments_angle(segments, tolerance=5):
    """
    Snap every segment's orientation to the nearest standard angle.

    Args:
        segments: Segments as [x1, y1, x2, y2].
        tolerance: Maximum degrees of difference allowed to snap (passed to snap_angle).

    Returns:
        Segments with angle-snapped endpoints.
    """
    snapped = []
    for segment in segments:
        x1 = segment[0]
        y1 = segment[1]
        x2 = segment[2]
        y2 = segment[3]
        angle = np.degrees(np.atan2(y2 - y1, x2 - x1)) # get angle
        corrected_angle = snap_angle(angle, tolerance)

        if corrected_angle != angle:
            # recompute endpoint based on snapped angle
            length = np.sqrt((x2 - x1) ** 2 + (y2 - y1) ** 2)
            angle_radians = np.radians(corrected_angle)
            # rotate around the first endpoint
            x2 = int(x1 + length * np.cos(angle_radians))
            y2 = int(y1 + length * np.sin(angle_radians))

        snapped.append([x1, y1, x2, y2])

    return snapped


**Below:** three concrete cases. A segment's raw `atan2` angle is compared against `[0, 45, 90, 135, 180]`; if the closest one is within tolerance, the second endpoint rotates around the first to land exactly on it (cases 1-2). Otherwise the segment is left as-is, angle and all (case 3) - a genuinely diagonal wall isn't forced onto a grid it doesn't belong to.

![snap_angle cases](NB%20IMAGES/03c_snap_angle.png)

### `bridge_wall_gaps` - fusing fragments of the same wall across a small gap

Collapses the many short, overlapping/collinear segment fragments that Hough or LSD produce along a single physical wall into one segment spanning the whole wall.

1. **Angle bucketing**: every segment's orientation is folded into `[0, 180)` and rounded to the nearest integer degree; only segments already sharing a bucket are ever considered.
2. **Union-find over endpoint-adjacency**, within a bucket: two segments are linked only if the closest pair of their endpoints is within `max_gap`, and that endpoint's perpendicular offset from the other segment's supporting line is within `max_perp_offset`. This second check rejects fusing two merely-nearby but sideways-offset parallel walls into one fake wall.
3. Each resulting connected component is merged into one segment: endpoints projected onto the bucket's direction vector, keeping the two extremes.

In [50]:
def bridge_wall_gaps(segments, max_gap, max_perp_offset=None):
    """
    Fuse same-orientation segments separated by a small gap.

    Args:
        segments: Segments as [x1, y1, x2, y2].
        max_gap: Maximum gap, in pixels, to bridge.
        max_perp_offset: Maximum perpendicular offset allowed between
                         the two segments' lines. Defaults to max_gap.

    Returns:
        Segments with small gaps bridged.
    """
    # fuses only small, same-line fragmentation gaps
    if max_perp_offset is None:
        max_perp_offset = max_gap

    n = len(segments)
    parent = list(range(n))

    # find the root
    def find(i):
        while parent[i] != i:
            parent[i] = parent[parent[i]]
            i = parent[i]
        return i

    # unify the two segments
    def union(i, j):
        ri, rj = find(i), find(j)
        if ri != rj:
            parent[ri] = rj

    # map segments to [0, 180) since start->end vs end->start describes the same undirected wall orientation
    angles = [round(np.degrees(np.atan2(y2 - y1, x2 - x1)) % 180) for x1, y1, x2, y2 in segments]

    # pairwise comparison of segments to find those that are close enough to bridge
    for i in range(n):
        xi1, yi1, xi2, yi2 = segments[i]
        angle_rad = np.radians(angles[i])
        direction = np.array([np.cos(angle_rad), np.sin(angle_rad)])
        normal = np.array([-direction[1], direction[0]])
        first_endpoint = np.array([xi1, yi1])

        for j in range(i + 1, n):
            if angles[j] != angles[i]:  # only bridge segments already on the same orientation
                continue
            
            xj1, yj1, xj2, yj2 = segments[j]
            endpoints_i = [(xi1, yi1), (xi2, yi2)]
            endpoints_j = [(xj1, yj1), (xj2, yj2)]

            # find the closest distance between any two endpoints of the two segments
            closest = min(
                np.hypot(pi[0] - pj[0], pi[1] - pj[1])
                for pi in endpoints_i for pj in endpoints_j
            )
            if closest > max_gap:
                continue

            # perpendicular offset between the two lines
            # project the vector from the first endpoint of segment i to the first endpoint of segment j onto the normal of segment i
            perp_offset = abs(np.dot(np.array([xj1, yj1]) - first_endpoint, normal)) 
            if perp_offset <= max_perp_offset:
                union(i, j)

    # the result is a set of clusters of segments that are likely to be part of the same wall
    
    clusters = {}
    for i in range(n):
        clusters.setdefault(find(i), []).append(i) # dict: root -> list of indices of segments in that cluster

    bridged = []
    for indices in clusters.values():
        cluster_segments = [segments[i] for i in indices] # list of segments in this cluster
        if len(cluster_segments) == 1:
            bridged.append(list(cluster_segments[0]))
            continue

        bucket = angles[indices[0]] # angle of the segments in the bucket (same for all)
        angle_rad = np.radians(bucket)
        direction = np.array([np.cos(angle_rad), np.sin(angle_rad)])

        # collect all endpoints (initial and final) of the segments in the cluster
        points = np.array(
            [[s[0], s[1]] for s in cluster_segments] +
            [[s[2], s[3]] for s in cluster_segments]
        )

        # matrix multiplication
        # projection onto the 1D axis defined by the segment direction
        # (how much do I have to move along the segment direction to reach the projection of each point)
        projections = points @ direction
        # measure the nearest and farthest endpoints on the direction line
        # will be the new endpoints of the merged segment
        p_min = points[np.argmin(projections)]
        p_max = points[np.argmax(projections)]

        bridged.append([int(p_min[0]), int(p_min[1]), int(p_max[0]), int(p_max[1])])

    return bridged


**Below:** the four situations this function actually has to tell apart. Only case 1 (and the collinear half of case 3) gets fused - same orientation bucket, small gap. Cases 2 and 4 both have endpoints close together yet are correctly left unbridged: case 2 because the two segments point in different directions (a real corner, not a detection artifact), case 4 because same-direction segments that are offset sideways are two distinct nearby walls, not one fragmented wall. Getting cases 2 and 4 wrong would silently merge unrelated walls or erase real corners.

![bridge_wall_gaps cases](NB%20IMAGES/03d_bridge_wall_gaps.png)

### `snap_corners` - closing gaps at wall junctions

After merging, wall corners are often still slightly misaligned: two segments meant to meet at exactly the same point end up a few pixels apart because they were extracted independently. `snap_corners` cleans this up:

1. Flatten every segment into its two endpoints (`2 * len(segments)` points total).
2. For each not-yet-processed endpoint, find every other endpoint within `snap_distance` (Euclidean). If exactly two distinct segments meet in that cluster, snap both endpoints to the intersection of the two segments' lines, so the corner lands exactly where the two walls cross and neither segment's angle gets disturbed. Otherwise (three or more segments meeting, e.g. a T-junction), average all their positions together, including the anchor point itself, same as before.
3. Store that snapped position for every point in the cluster, so all nearby endpoints, potentially belonging to several different segments meeting at the same corner, converge to one shared coordinate.
4. Rebuild the segment list, replacing each original endpoint with its snapped position (falling back to the original coordinate if a point was never within range of anything else).

A genuinely dangling endpoint (no other endpoint nearby at all, cluster of one) is no longer left untouched: it's projected onto the nearest other segment's line instead, `fix_broken_corners`'s logic, folded in directly.

In [51]:
def snap_corners(segments, snap_distance=15):
    """
    Snap nearby segment endpoints together so wall corners meet cleanly,
    and project genuinely dangling endpoints (no other endpoint nearby at
    all) onto the nearest other segment's line.

    Args:
        segments: Segments as [x1, y1, x2, y2].
        snap_distance: Maximum distance, in pixels, both to snap endpoints
                        together and to project a dangling endpoint onto
                        another segment's line.

    Returns:
        Segments with corner-snapped and dangling endpoints resolved.
    """

    # reason on endpoints distance
    endpoints = []
    for segment in segments:
        endpoints.append((segment[0], segment[1]))  # start point
        endpoints.append((segment[2], segment[3]))  # end point

    def segment_direction(seg_idx):
        x1, y1, x2, y2 = segments[seg_idx]
        direction = np.array([x2 - x1, y2 - y1], dtype=float)
        norm = np.hypot(*direction)
        return direction / norm if norm > 1e-9 else np.array([1.0, 0.0])

    def line_intersection(seg_a, seg_b):
        # write each segment as an infinite line in point-direction form:
        #   line A: P(t) = p_a + t * d_a
        #   line B: Q(s) = p_b + s * d_b
        p_a = np.array(segments[seg_a][:2], dtype=float)
        d_a = segment_direction(seg_a)
        p_b = np.array(segments[seg_b][:2], dtype=float)
        d_b = segment_direction(seg_b)

        # the two lines cross where P(t) == Q(s). Taking the 2D cross product
        # of both sides with d_b eliminates s (cross(d_b, d_b) == 0), leaving
        # one equation in the single unknown t:
        #   cross(p_a - p_b, d_b) + t * cross(d_a, d_b) = 0
        # where cross(u, v) = u.x * v.y - u.y * v.x

        cross_da_db = d_a[0] * d_b[1] - d_a[1] * d_b[0]  # = sin(angle between the lines)
        if abs(cross_da_db) < 1e-9:
            # lines are parallel (or the same line): no single intersection point
            return None

        p_diff = p_b - p_a
        cross_pdiff_db = p_diff[0] * d_b[1] - p_diff[1] * d_b[0]
        t = cross_pdiff_db / cross_da_db

        # step distance t along line A from p_a: by construction this point
        # also satisfies line B's equation, so it is the exact intersection
        intersection = p_a + t * d_a
        return intersection

    # return the closest projection of the segment onto
    # other segments within snap_distance, or None.
    # used for segments that may intersect another segment not at their endpoint (ex T junction)
    def project_onto_other_segments(px, py, own_seg_idx):
        best = None
        for j, (ox1, oy1, ox2, oy2) in enumerate(segments):
            if j == own_seg_idx:
                continue
            dx, dy = ox2 - ox1, oy2 - oy1
            length = np.hypot(dx, dy)
            if length == 0:
                continue
            direction = np.array([dx, dy]) / length

            # project (px, py) onto the segment
            t = np.dot(np.array([px - ox1, py - oy1]), direction)
            t_clamped = max(-snap_distance, min(length + snap_distance, t))
            projected = np.array([ox1, oy1]) + t_clamped * direction

            dist = np.hypot(px - projected[0], py - projected[1])
            if dist <= snap_distance and (best is None or dist < best[0]):
                best = (dist, projected)
        return best

    # for each endpoint, find nearby endpoints and snap them
    snapped_points = {}
    for i, (x1, y1) in enumerate(endpoints):
        if i in snapped_points:
            continue
        cluster = [(x1, y1)]
        cluster_indices = [i]
        for j, (x2, y2) in enumerate(endpoints):
            if i != j and np.hypot(x2 - x1, y2 - y1) < snap_distance:
                cluster.append((x2, y2))
                cluster_indices.append(j)

        # segments contributing an endpoint to this cluster
        cluster_segments = set(k // 2 for k in cluster_indices)

        if len(cluster_segments) == 1:
            # no other endpoint nearby at all
            own_seg_idx = i // 2
            fallback = project_onto_other_segments(x1, y1, own_seg_idx)
            if fallback is not None:
                _, projected = fallback
                snap_x, snap_y = int(round(projected[0])), int(round(projected[1]))
            else:
                snap_x, snap_y = x1, y1
                
            snapped_points[i] = (snap_x, snap_y)
            continue

        corner = None
        if len(cluster_segments) == 2:
            corner = line_intersection(*cluster_segments)

        if corner is not None: # 2 segments
            snap_x, snap_y = int(round(corner[0])), int(round(corner[1]))
        else:
            # compute average position of the cluster
            snap_x = int(sum(p[0] for p in cluster) / len(cluster))
            snap_y = int(sum(p[1] for p in cluster) / len(cluster))

        # store the snapped position for ALL points in the cluster
        snapped_points[i] = (snap_x, snap_y)
        for j, (x2, y2) in enumerate(endpoints):
            if i != j and np.hypot(x2 - x1, y2 - y1) < snap_distance:
                snapped_points[j] = (snap_x, snap_y)

    # rebuild segments with snapped endpoints
    result = []
    for idx, segment in enumerate(segments):
        p1 = snapped_points.get(idx * 2, (segment[0], segment[1]))
        p2 = snapped_points.get(idx * 2 + 1, (segment[2], segment[3]))
        result.append([p1[0], p1[1], p2[0], p2[1]])

    return result


**Below:** the three shapes an endpoint cluster can take, and what `snap_corners` does with each. Two endpoints (case 1) snap to the exact line-line intersection, so the corner's angle isn't disturbed by averaging. Three or more (case 2, a T-junction) fall back to an averaged position, since there's no single well-defined intersection for three lines. A cluster of one - nothing else within `snap_distance` (case 3) - is no longer left dangling: it's projected onto the nearest other segment's line instead, closing the gap without disturbing that other wall.

![snap_corners cases](NB%20IMAGES/03e_snap_corners.png)

### `regularize` / `draw_regularized` - the full post-processing composition

`regularize` chains three functions in a fixed order, each with a resolution-proportional parameter:

```
snap_segments_angle()
    -> bridge_wall_gaps()
        -> snap_corners()
```

The order matters: angles are straightened first so that `bridge_wall_gaps`'s angle-bucketing operates on already-canonical orientations, reducing the chance that near-45-but-not-quite segments fall into the wrong bucket; segments are bridged before corner-snapping so that snapping operates on fewer, longer, more representative endpoints rather than on every fragment's endpoint individually. `snap_corners` also snaps genuinely dangling endpoints by projecting them onto the nearest other segment's line.

![regularization stages](NB%20IMAGES/04_regularization_stages_visual.png)

In [52]:
def regularize(segments, img_shape, angle_tolerance=10, merge_distance_frac=0.01, corner_snap_frac=0.003):
    """
    Run the full regularization pipeline: angle snapping, gap bridging, and corner snapping.

    Args:
        segments: Raw detected segments as [x1, y1, x2, y2].
        img_shape: (height, width) of the source image.
        angle_tolerance: Degrees of tolerance for snap_segments_angle.
        merge_distance_frac: bridge_wall_gaps' max_gap, as a fraction of min(h,w).
        corner_snap_frac: snap_corners' snap distance, as a fraction of min(h,w).

    Returns:
        Regularized segments.
    """
    h, w = img_shape

    # 1. snap angles
    snapped = snap_segments_angle(segments, angle_tolerance)

    # 2. bridge collinear fragments split by a small gap (see bridge_wall_gaps below)
    bridged = bridge_wall_gaps(snapped, max_gap=int(min(h,w) * merge_distance_frac))

    # 3. snap corners, including dangling endpoints
    final = snap_corners(bridged, snap_distance=int(min(h,w) * corner_snap_frac))

    return final


def draw_regularized(segments, img_shape):
    """
    Draw a segment list onto a black canvas for a quick visual check.

    Args:
        segments: Segments as [x1, y1, x2, y2].
        img_shape: (height, width) of the canvas.

    Returns:
        The canvas with each segment drawn.
    """
    h, w = img_shape
    canvas = np.zeros((h, w, 3), dtype=np.uint8)
    for x1, y1, x2, y2 in segments:
        cv2.line(canvas, (x1, y1), (x2, y2), (0, 255, 0), 2)

    plt.imshow(canvas, cmap='gray')
    return canvas


## CAD-ready export: wall thickness

Everything above (`regularize`) produces clean, straight, zero-width centerlines: good enough for `compute_metrics`'s vector-space comparison against the CVC-FP ground truth, but not directly usable as a CAD deliverable. `regularize()` now also bridges gaps and snaps dangling endpoints itself (via `bridge_wall_gaps`/`snap_corners`, above), so what's left here is wall thickness.

The functions below add that: recover each wall's thickness from the pixel data already computed by `isolate_walls_multiscale`, so the exported geometry is a set of filled wall footprints instead of zero-width lines. `estimate_wall_thickness`/`segments_to_svg_walls` are deliberately not wired into `evaluate_pair()` (`compute_metrics` only scores centerlines); they only run in the ad-hoc single-image export cell further down.

### `estimate_wall_thickness` - recovering per-wall thickness

Reuses the same distance-transform idea already central to `isolate_walls`/`isolate_walls_multiscale` replaces every wall pixel with its distance to the nearest background pixel, so a point near a stroke's centerline reads roughly `thickness / 2`. Recomputing it locally here, rather than threading it out of `isolate_walls_multiscale`, keeps that function's signature untouched.

For each final segment:

- `n_samples` points are taken evenly along its length, excluding both endpoints, since endpoints sit at corners/T-junctions where two walls overlap and the distance transform reads artificially thick right there.
- At each sample, thickness is approximately `2 * dist[y, x]`. Samples that land off the wall mask entirely (`dist ~ 0`, possible since `regularize()`'s angle/corner snapping can shift a centerline slightly) are discarded.
- The median of the remaining samples is used, not the mean, for robustness against a single sample landing near a doorway notch or other local irregularity. If every sample was discarded, a small resolution-proportional default is used instead of crashing or returning zero.

Returns a plain list of thickness values (in pixels), parallel to `segments`, with no change to the segment format itself.


In [53]:
def estimate_wall_thickness(segments, walls_mask, n_samples=7):
    """
    Estimate each segment's wall thickness from the binary wall image.

    Args:
        segments: Wall centerlines as [x1, y1, x2, y2].
        walls_mask: Binary wall mask the segments were detected from.
        n_samples: Number of sample points per segment.

    Returns:
        Estimated thickness in pixels, one per segment.
    """

    dist = cv2.distanceTransform(walls_mask, cv2.DIST_L2, 5) # walls_mask = bynary walls img
    h, w = walls_mask.shape
    default_thickness = max(2, int(0.003 * min(h, w))) # always proportional to img size

    thicknesses = []
    for x1, y1, x2, y2 in segments:
        samples = []
        # sample along the segment
        for t in np.linspace(0, 1, n_samples + 2)[1:-1]: # exlude the first and last samples (endpoints)
            # compute the samples
            x = int(round(x1 + t * (x2 - x1)))
            y = int(round(y1 + t * (y2 - y1)))

            if 0 <= y < h and 0 <= x < w: # bounds check
                d = dist[y, x] # combine the distance mask and the extracted segments positions
                if d > 0.5: # discard samples that missed the wall mask entirely
                    samples.append(2 * d)  # 2*distance

        thicknesses.append(float(np.median(samples)) if samples else float(default_thickness))

    return thicknesses


### `segments_to_svg_walls` - thickness-aware CAD export

This is where the pipeline produces the stated output: a real `.svg` file, not a raster image.
This writes each wall as a filled rectangle instead of a zero-width `<line>`: the segment's unit perpendicular vector is used to offset each endpoint by `thickness / 2` on both sides, giving the four corners of the wall's footprint, written as one SVG `<polygon>` per wall.

In [54]:
def segments_to_svg_walls(segments, thicknesses, shape, output_path):
    """
    Write wall segments to an SVG as filled rectangles using their thickness.

    Args:
        segments: Segments as [x1, y1, x2, y2].
        thicknesses: Per-segment wall thickness in pixels.
        shape: (height, width) of the source image.
        output_path: File path to write the SVG to.

    Returns:
        output_path.
    """
    h, w = shape

    polygons = []
    for (x1, y1, x2, y2), thickness in zip(segments, thicknesses):
        dx, dy = x2 - x1, y2 - y1
        length = np.hypot(dx, dy)
        if length == 0:
            continue

        nx, ny = -dy / length, dx / length  # unit perpendicular to the wall
        half_thickness = thickness / 2

        # compute points of the wall with thickness
        p1 = (x1 + nx * half_thickness, y1 + ny * half_thickness)
        p2 = (x2 + nx * half_thickness, y2 + ny * half_thickness)
        p3 = (x2 - nx * half_thickness, y2 - ny * half_thickness)
        p4 = (x1 - nx * half_thickness, y1 - ny * half_thickness)

        points = " ".join(f"{px:.1f},{py:.1f}" for px, py in (p1, p2, p3, p4))
        polygons.append(f'  <polygon points="{points}" fill="black" />')

    svg_body = "\n".join(polygons)
    svg = f'<?xml version="1.0" encoding="UTF-8"?>\n<svg xmlns="http://www.w3.org/2000/svg" width="{w}" height="{h}" viewBox="0 0 {w} {h}">\n{svg_body}\n</svg>\n'

    with open(output_path, "w") as f:
        f.write(svg)

    # display the svg as text
    with open(output_path, "r") as f:
        print(f.read())

    return output_path


In [55]:
#img = cv2.imread(input("Enter the path to the images to preprocess: "))
#binary_img = preprocess(img)
#walls_mask = isolate_walls_multiscale(binary_img)
#raw_segments = detect_walls_lsd(walls_mask)
#regularized = regularize(raw_segments, img.shape[:2])
#thicknesses = estimate_wall_thickness(regularized, walls_mask)
#canvas = draw_regularized(regularized, img.shape[:2])
#svg = segments_to_svg_walls(regularized, thicknesses, img.shape[:2], "output.svg")


**Below:** the ad hoc CAD-ready export cell above, run end to end on a sample floorplan. Left: the regularized wall network with recovered thickness, rendered as filled footprint polygons (the same geometry `segments_to_svg_walls` writes to `.svg`) rather than zero-width centerlines. Right: the same footprints composited back over the original scan, to check by eye that each polygon actually sits on top of the wall it was detected from.

![CAD-ready export](NB%20IMAGES/06_cad_export.png)

## Verification: wall extraction vs. ground truth (CVC-FP)

Ground-truth walls in CVC-FP are filled polygons (the wall's true footprint), while predicted segments are thin line approximations of the detected wall mask's edges, not a centerline. Comparing them exactly would fail even on a perfect detection, so a tolerance is needed, but doing that tolerance-matching by rasterizing to a pixel mask, skeletonizing, and dilating with a square kernel introduces its own error. Since the actual deliverable here is vector geometry, comparison is done in vector space:

1. Each GT `Wall` polygon in CVC-FP is an unclosed 4-point quad, the wall's rectangular footprint, not a line. Its centerline is computed exactly: of the two pairs of opposite edges, the shorter pair are the wall's end caps, and the centerline is the segment joining their midpoints (`wall_centerline`). No skeletonization needed.
2. Predicted segments are used directly as line geometry.
3. Both the GT centerline network and the predicted line network are buffered by a tolerance `tau` (resolution-proportional, same convention used for other thresholds in this notebook) directly in continuous coordinates, to absorb the edge-vs-centerline offset and normal annotation noise.
4. **Precision** ("correctness") = fraction of predicted line length that lies inside the buffered GT network: how much of what was drawn is a real wall.
5. **Recall** ("completeness") = fraction of GT centerline length that lies inside the buffered predicted network: how much of the true wall network was found.
6. **F1** = harmonic mean of the two.

### Evaluation setup

- `xml.etree.ElementTree` parses the CVC-FP ground-truth `.svg` files directly as XML.
- `shapely.geometry.LineString` / `shapely.ops.unary_union` provide the vector-space geometry primitives.
- `SVG_NS`: CVC-FP's SVGs declare a default XML namespace (`xmlns="http://www.w3.org/2000/svg"`), which means a naive `root.find("polygon")` silently matches nothing, since ElementTree requires namespace-qualified tag names once a default namespace is declared.


In [56]:
import xml.etree.ElementTree as ET # to parse ground-truth SVGs (stdlib, no new dependency)
import glob
from shapely.geometry import LineString # vector-space wall geometry, buffering/intersection for GT comparison
from shapely.ops import unary_union

# CVC-FP SVGs declare a default xmlns, so every tag lookup below must be namespace-qualified
# (plain root.find("polygon") silently matches nothing)
SVG_NS = {"svg": "http://www.w3.org/2000/svg"}

### `parse_gt_walls` - reading CVC-FP ground truth

Extracts ground-truth wall geometry from a CVC-FP SVG. CVC-FP's SVGs store each annotated architectural element as an SVG `<polygon>` with a `class` attribute (`Wall`, `Door`, `Window`, `Room`, ...). This function:

1. Reads the custom top-level `<width>`/`<height>` tags, CVC-FP stores these as plain custom SVG tags rather than real `width`/`height` attributes on the `<svg>` root, and returns them as `declared_size`, used purely as a sanity check later in `evaluate_pair` to confirm the SVG and its paired PNG actually describe the same pixel grid.
2. Filters `<polygon class="Wall">` elements only; every other annotated class (doors, windows, room labels) is ignored, since this pipeline's scope is wall geometry.
3. Parses each polygon's `points` attribute (`"x1,y1 x2,y2 x3,y3 ..."`, space-separated pairs, comma-separated coordinates) into a list of `(x, y)` tuples.

Returns `(polygons, declared_size)`: the raw list of wall footprint polygons (still full quads, not yet reduced to centerlines) plus the declared canvas size.


In [57]:
# return points of the extreme vertices of the walls in the SVG
def parse_gt_walls(svg_path):
    """
    Parse wall polygons and declared image size from a ground-truth SVG.

    Args:
        svg_path: Path to a CVC-FP ground-truth SVG file.

    Returns:
        Wall polygons and declared (width, height).
    """
    root = ET.parse(svg_path).getroot()

    # CVC-FP stores width/height as plain custom tags, not real svg attributes
    # here only as a sanity check that the svg/image pair actually match
    width_tag = root.find("svg:width", SVG_NS)
    height_tag = root.find("svg:height", SVG_NS)

    if width_tag is not None and height_tag is not None:
        declared_size = (int(width_tag.text), int(height_tag.text))  
    else:
        declared_size = None

    polygons = []
    for poly in root.findall("svg:polygon", SVG_NS):
        if poly.get("class") != "Wall": # consider only walls polygons
            continue
        # points = 1.123,2.345 3.456,4.567 ... -> get single points by splitting on space and then on comma
        points = [tuple(map(float, pair.split(","))) for pair in poly.get("points").split()]
        polygons.append(points)

    return polygons, declared_size

### `rasterize_wall_polygons` - GT polygons to pixel mask (visualization only)

`cv2.fillPoly` rasterizes each GT wall polygon (filled, not outlined) onto a blank `uint8` mask matching the image's shape.

In [58]:
# used only for the visual overlay in visualize_wall_match
def rasterize_wall_polygons(polygons, shape):
    """
    Rasterize ground-truth wall polygons into a binary mask.

    Args:
        polygons: Wall polygons, as returned by parse_gt_walls.
        shape: (height, width) of the mask.

    Returns:
        Binary mask (0/255).
    """
    mask = np.zeros(shape, dtype=np.uint8) # mask of 0s with the same shape as the image

    for points in polygons:
        pts = np.array(points, dtype=np.int32).reshape((-1, 1, 2)) # to have fillPoly expected shape
        cv2.fillPoly(mask, [pts], 255) # color mask with white where the polygon is filled (pts)
    return mask

### `wall_centerline` - exact centerline of a wall footprint polygon

A GT `Wall` polygon is the wall's footprint, a thin rectangle (4 points: two long edges are the wall's length, two short edges are its thickness), not a line. To compare it fairly against a predicted line segment, it must be reduced to a single representative axis.

- **Quadrilateral case (`n == 4`, the common case)**: compute the length of all 4 edges, identify which pair of opposite edges is shorter (`edges[0]+edges[2]` vs. `edges[1]+edges[3]`), the shorter pair are the wall's short end caps, the longer pair run along the wall's length. The centerline is the segment joining the midpoints of the two short edges. This recovers the exact geometric centerline of a rectangle regardless of its rotation, no assumption of axis-alignment is needed.
- **Fallback (`n != 4`, irregular/L-shaped footprints)**: approximate the long axis by finding the two mutually most-distant vertices (brute-force over all vertex pairs) and use them directly as the centerline endpoints. This is an approximation, not an exact centerline, but handles the rare non-quad annotation gracefully instead of crashing.

This function is the geometric core that makes the whole evaluation comparison well-posed: without it, comparing a filled polygon against a thin predicted line would always show poor overlap even for a perfect detection, since a line has zero area and a polygon does not.


In [59]:
# return the centerline, line that joins the midpoints of the two smaller edges of the wall polygon
def wall_centerline(points):
    """
    Reduce a wall polygon to its centerline (start, end) points.

    Args:
        points: Polygon vertices, in order.

    Returns:
        (centerline_start, centerline_end).
    """
    pts = np.array(points, dtype=float)
    n = len(pts)

    if n == 4: # should be quadrilateral, but just in case check anyway
        edges = [(pts[k], pts[(k + 1) % 4]) for k in range(4)] # store the edges (pair of consecutive vertices)
        edge_lengths = [np.sqrt((b[0] - a[0])**2 + (b[1] - a[1])**2) for a, b in edges] # compute the length of each edge

        # find the shorter pair of opposite edges
        if edge_lengths[0] + edge_lengths[2] <= edge_lengths[1] + edge_lengths[3]:
            shorter_edges = (edges[0], edges[2])
        else:
            shorter_edges = (edges[1], edges[3])

        # compute the midpoints
        centerline_start = (shorter_edges[0][0] + shorter_edges[0][1]) / 2
        centerline_end = (shorter_edges[1][0] + shorter_edges[1][1]) / 2

    else: # if the polygon is not a quadrilateral --> the two most distant vertices approximate the long axis
        # distances = [dist, i, j]
        distances = [(np.sqrt((pts[i][0] - pts[j][0])**2 + (pts[i][1] - pts[j][1])**2), i, j) for i in range(n) for j in range(i + 1, n)]
        _, i, j = max(distances) # get the indexes of the two most distant vertices (the distance is not relevant)
        centerline_start, centerline_end = pts[i], pts[j]

    return tuple(centerline_start), tuple(centerline_end)

### `compute_metrics` - precision, recall, F1

Computes precision/recall/F1 between predicted wall lines and the GT wall network. Given predicted segments and GT wall polygons for one image:

1. **Tolerance**: if not explicitly passed, `tolerance = max(3, round(tolerance_frac * min(h, w)))` pixels, resolution-proportional, floored at 3px so small images still get a workable buffer. `tolerance_frac` defaults to 0.005 (for the running example, 905x1094, this evaluates to 5px) but is a tunable parameter of `compute_metrics`.
2. **Build networks**: GT polygons are reduced to centerlines (`wall_centerline`) and wrapped as Shapely `LineString`s; predicted segments become `LineString`s directly. `unary_union` merges each list into one combined geometry representing the whole wall network as a single object.
3. **Buffer**: `.buffer(tolerance)` inflates each 1-D line network into a 2-D tolerance corridor of width `2 * tolerance` around every line, which is what absorbs the edge-vs-centerline offset (predicted segments trace wall edges, GT centerlines run through the wall's middle) and normal annotation/detection noise.
4. **Precision** = `length(pred_network intersect gt_corridor) / length(pred_network)`: the fraction of predicted line length that falls within a plausible margin of a real wall.
5. **Recall** = `length(gt_network intersect pred_corridor) / length(gt_network)`: the fraction of real wall length that was found. Missing an entire wall, or finding only part of one, lowers this proportionally to how much was missed, not all-or-nothing per wall.
6. **F1** = harmonic mean of the two, for a single scalar to rank configurations by.

Because both metrics are length-weighted rather than counting whole segments/lines, a wall that is correctly detected for only half its length contributes exactly half credit, a materially fairer metric than a naive per-segment match/no-match count would give.


In [60]:
# compute precision, recall, and F1 score (line intersection approach, not pixel by pixel)
def compute_metrics(pred_segments, gt_polygons, img_shape, tolerance_frac=0.0025):
    """
    Compute precision, recall, and F1 of predicted walls against ground truth.

    Args:
        pred_segments: Predicted segments as [x1, y1, x2, y2].
        gt_polygons: Ground-truth wall polygons.
        img_shape: (height, width) of the source image.
        tolerance: Matching tolerance in pixels. Defaults to a
            resolution-proportional value derived from tolerance_frac.
        tolerance_frac: Fraction of min(h, w) used to derive the default
            tolerance when tolerance is not explicitly passed.

    Returns:
        precision, recall, f1, and tolerance used.
    """
    h, w = img_shape
    tolerance = max(3, round(tolerance_frac * min(h, w))) # resolution-proportional

    # convert ground-truth polygons to centerline segments
    gt_lines = [LineString(wall_centerline(poly)) for poly in gt_polygons if len(poly) >= 3]

    # convert predicted segments to LineStrings
    pred_lines = [LineString([(x1, y1), (x2, y2)]) for x1, y1, x2, y2 in pred_segments]

    # compute a single geometry for the entire network of walls (union of all lines)
    gt_network = unary_union(gt_lines) if gt_lines else LineString()
    pred_network = unary_union(pred_lines) if pred_lines else LineString()

    # get a geometry that represents the "tolerance zone"
    gt_with_tolerance = gt_network.buffer(tolerance)
    pred_with_tolerance = pred_network.buffer(tolerance)

    pred_total_length = sum(line.length for line in pred_lines)
    gt_total_length = sum(line.length for line in gt_lines)

    # precision = TP / (TP + FP)
    # (length of predicted walls that intersect with the ground truth tolerance zone) / (total length of predicted walls)
    precision = pred_network.intersection(gt_with_tolerance).length / pred_total_length if pred_total_length != 0 else 0.0

    # recall = TP / (TP + FN)
    # (length of ground truth walls that intersect with the predicted tolerance zone) / (total length of ground truth walls)
    recall = gt_network.intersection(pred_with_tolerance).length / gt_total_length if gt_total_length != 0 else 0.0

    # F1 score = 2 * (precision * recall) / (precision + recall)
    f1 = 0.0 if (precision + recall) == 0 else 2 * precision * recall / (precision + recall)        

    return {
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "tolerance": tolerance
    }

### `visualize_wall_match` - GT vs prediction overlay

Renders the predicted segments as a 2px-thick raster mask (thicker than the 1px-conceptual line, purely for it to be visible) and composites it against the GT raster mask (from `rasterize_wall_polygons`) into one BGR image:

- GT mask -> blue channel
- Predicted mask -> red channel
- Wherever both are white, the additive color mix renders as magenta, an at-a-glance agreement color, with pure blue meaning a missed GT wall (false negative) and pure red meaning a wrong prediction (false positive).

![verification overlay](NB%20IMAGES/05_verification_overlay.png)

In [61]:
def visualize_wall_match(img, pred_segments, gt_mask, metrics, visualize=False):
    """
    Print match metrics and build a ground-truth-vs-predicted overlay for inspection.

    Args:
        img: Original BGR image.
        pred_segments: Predicted segments as [x1, y1, x2, y2].
        gt_mask: Rasterized ground-truth mask.
        metrics: Metrics dict from compute_metrics.

    Returns:
        None.
    """
    h, w = gt_mask.shape

    pred_mask = np.zeros((h, w), dtype=np.uint8)

    for x1, y1, x2, y2 in pred_segments:
        cv2.line(pred_mask, (int(x1), int(y1)), (int(x2), int(y2)), 255, 2) # thicker line just to make it visible

    # colour overlay in BGR (same convention show_images expects)
    overlay = np.zeros((h, w, 3), dtype=np.uint8)
    # overlap is "automatically" displayed as magenta (red + blue) where the two masks intersect
    overlay[..., 0] = gt_mask   # B channel
    overlay[..., 2] = pred_mask # R channel

    print(f"precision={metrics['precision']:.3f}  recall={metrics['recall']:.3f}  f1={metrics['f1']:.3f}  tolerance={metrics['tolerance']}px")

    if visualize:
        show_images({
            "Original": img,
            "GT walls": gt_mask,
            "Predicted lines": pred_mask,
            "Overlay (GT=blue, Pred=red)": overlay
        })

### `pipeline`

function that determines the steps of the pipeline

In [62]:
def pipeline(img, gt_polygons, h, w, visualize=False):
    """
    Run the full pipeline

    Args:
        img: Original BGR image.
        gt_polygons: Ground-truth wall polygons.
        h: Height of the image.
        w: Width of the image.
        visualize: Whether to visualize the intermediate steps imgs.
    
    Returns:
        regularized_img: The final regularized wall segments.
        gt_mask: The rasterized ground-truth wall mask.
        metrics: The computed precision, recall, and F1 score.
    """

    # lsd pipeline
    #binary_img = preprocess(img)
    #walls_mask = isolate_walls(binary_img)
    #pred_segments = detect_walls_lsd(walls_mask)
    #regularized_img = regularize(pred_segments, img.shape[:2])

    # hough pipeline
    bynary_img = preprocess(img, visualize=visualize)
    walls_mask = isolate_walls(bynary_img, visualize=visualize)
    hough_img = detect_walls(walls_mask, visualize=visualize)
    regularized_img = regularize(hough_img, img.shape[:2])
    
    gt_mask = rasterize_wall_polygons(gt_polygons, (h, w),) # for visualize_wall_match's overlay only
    metrics = compute_metrics(regularized_img, gt_polygons, (h, w))

    return regularized_img, gt_mask, metrics

### `evaluate_pair` - the shared scoring entry point

Single-image and batch entry point, shared by `verify_single` and `verify_dataset`: runs the pipeline on one `(image, GT SVG)` pair and scores the result.

1. Load the image, parse the GT SVG.
2. **Sanity check**: if the SVG declares a `width`/`height` and it doesn't match the actual loaded image's `(w, h)`, raise a `ValueError` immediately rather than silently scoring against a mismatched coordinate system. This is what confirms CVC-FP's SVGs share a 1:1 pixel grid with their PNGs.
3. **The scored path**: `preprocess -> isolate_walls_multiscale -> detect_walls (Hough) -> regularize`. A parallel LSD-based path (`detect_walls_lsd`, no `regularize` call) is present in the cell but commented out, kept for quick manual comparison.
4. Build the GT raster mask (`rasterize_wall_polygons`, for `visualize_wall_match`'s overlay only) and compute the vector-space metrics (`compute_metrics`).
5. Return `(metrics, regularized_segments, gt_mask, img)`: everything both `verify_single` (which also visualizes) and `verify_dataset` (which only aggregates numbers) need.

In [63]:
def evaluate_pair(img_path, svg_path, visualize=False):
    """
    Run the pipeline on one image/ground-truth pair and score the result.

    Args:
        img_path: Path to the raster floorplan image.
        svg_path: Path to the matching ground-truth SVG.

    Returns:
        metrics, regularized segments, rasterized GT mask, original image.
    """
    img = cv2.imread(img_path)
    if img is None:
        raise FileNotFoundError(f"Could not read image: {img_path}")

    polygons, declared_size = parse_gt_walls(svg_path)
    
    h, w = img.shape[:2]
    if declared_size is not None and declared_size != (w, h):
        raise ValueError(f"{svg_path} declares size {declared_size} but {img_path} is {(w, h)}")

    regularized_img, gt_mask, metrics = pipeline(img, polygons, h, w, visualize=visualize)
    
    return metrics, regularized_img, gt_mask, img

### `verify_single` - score one image, with visualization

Convenience wrapper for interactive use: if no `svg_path` is given explicitly, it derives it automatically from the image's filename via `glob(f"{img_name}_gt_*.svg")`, matching CVC-FP's naming convention where the ground-truth file encodes the room count in its suffix (e.g. `1.png` <-> `1_gt_14.svg` for a 14-room floorplan). Raises `FileNotFoundError` if no match exists, otherwise delegates to `evaluate_pair` and then calls `visualize_wall_match` to both print and (if uncommented) display the overlay figure. Returns the metrics dict so it can also be used programmatically, not just interactively.


In [64]:
# evaluate performances on a single image (works if names between original and svg are coherent)
def verify_single(img_path, svg_path=None, visualize=False):
    """
    Evaluate the pipeline on a single image, auto-discovering its ground-truth SVG.

    Args:
        img_path: Path to the raster floorplan image.
        svg_path: Path to the ground-truth SVG, or None to auto-discover it.
        visualize: Whether to show the intermediate pipeline stages and the GT-vs-prediction overlay.

    Returns:
        Metrics dict from compute_metrics.
    """
    if svg_path is None:
        img_name = os.path.splitext(img_path)[0]
        svg = glob.glob(f"{img_name}_gt_*.svg") # find the correspondent (name coherence) svg
        if not svg:
            raise FileNotFoundError(f"No ground-truth SVG found for {img_path} (expected {img_name}_gt_*.svg)")
        svg_path = svg[0]

    metrics, pred_segments, gt_mask, img = evaluate_pair(img_path, svg_path, visualize=visualize)
    visualize_wall_match(img, pred_segments, gt_mask, metrics, visualize=visualize)

    return metrics

### `verify_dataset` - batch evaluation across a whole folder

Iterates every supported image in `images_gt_dir` (sorted, for a deterministic run order), looks up its matching GT SVG by the same glob convention as `verify_single`, and skips with a printed warning any image that has none, rather than failing the whole batch. Unlike `verify_single`, it deliberately doesn't call `visualize_wall_match` per image, over CVC-FP's 122 scoreable images that would pop up over a hundred blocking matplotlib figures; only the numeric `metrics` from `evaluate_pair` are kept, one dict per image, appended to a `results` list.

Once all images are processed, it prints a quick summary, mean and median of precision/recall/f1 across the dataset, directly to stdout. `log_performance` (further down) turns this same `results` list into a persisted, comparable graph.

In [65]:
# evaluate performances at batch level (works if names between original and svg are coherent)
def verify_dataset(images_gt_dir):
    """
    Evaluate the pipeline over every image/ground-truth pair in a directory.

    Args:
        images_gt_dir: Directory of paired raster images and ground-truth SVGs.

    Returns:
        Metrics for each evaluated image.
    """
    supported_formats = (".png", ".jpg", ".jpeg")
    results = []

    # evaluate each image
    for filename in sorted(os.listdir(images_gt_dir)):
        if not filename.lower().endswith(supported_formats): # skip non supported formats
            continue

        img_path = os.path.join(images_gt_dir, filename)
        img_name = os.path.splitext(filename)[0]
        svg = glob.glob(os.path.join(images_gt_dir, f"{img_name}_gt_*.svg")) # find the correspondent svg img

        if not svg:
            print(f"skipping {filename}: no matching ground-truth SVG")
            continue

        metrics, *_ = evaluate_pair(img_path, svg[0])
        results.append({"image": filename, **metrics})

    if results:
        precisions = [r["precision"] for r in results]
        recalls = [r["recall"] for r in results]
        f1s = [r["f1"] for r in results]
        print(f"\nevaluated {len(results)} images")
        print(f"precision: mean={np.mean(precisions):.3f} median={np.median(precisions):.3f}")
        print(f"recall:    mean={np.mean(recalls):.3f} median={np.median(recalls):.3f}")
        print(f"f1:        mean={np.mean(f1s):.3f} median={np.median(f1s):.3f}")

    return results

## Performance tracking and graphs

`verify_dataset`/`verify_single` above compute precision/recall/F1 but only print them; the numbers are gone as soon as the pipeline or a parameter changes and the cell reruns. This section saves a graph plus a short text description of which pipeline/parameters produced it to `GRAPHS/` every time an evaluation is run, plus a history log so runs can be compared over time.

In [66]:
from datetime import datetime
import json
import re
import textwrap

GRAPHS_DIR = "GRAPHS"

# fixed categorical color per metric
METRIC_COLORS = {"precision": "#2a78d6", "recall": "#eb6834", "f1": "#1baf7a"}
CHART_SURFACE = "#fcfcfb"
CHART_INK = "#0b0b0b"
CHART_MUTED = "#898781"
CHART_GRID = "#e1e0d9"

### `log_performance` - persisting one evaluation run

Turns the in-memory `results` list from `verify_dataset` (or a single dict from `verify_single`, auto-wrapped into a one-element list) into three durable artifacts:

1. **`{run_id}_summary.png`**: a grouped bar chart, mean vs. median for each of precision/recall/f1.
2. **`{run_id}_metadata.json`**: the full record, including every per-image metric, not just the aggregate, auditable down to which specific image scored what.
3. **`GRAPHS/history.json`**: one compact aggregate-only entry appended (creating the file on first use), which `plot_history` reads to build the cross-run trend chart.


In [67]:
def log_performance(results, tag, description, output_dir=GRAPHS_DIR):
    """
    Save a summary graph and metadata for an evaluation run, and log it to history.

    Args:
        results: One or more metrics dicts to aggregate.
        tag: Short label for this run.
        description: Longer note describing this run.
        output_dir: Directory to write outputs to.

    Returns:
        The saved run metadata.
    """
    # accept list of results or a single result dict
    if isinstance(results, dict):
        results = [results]

    os.makedirs(output_dir, exist_ok=True)

    '''aggregate = { 
        "precision": { "mean": 0.8, "median": 0.8 },
        "recall": { "mean": 0.7, median": 0.7 },
        "f1": { "mean": 0.746, "median": 0.746 }
    }'''
    aggregate = {
        metric: {"mean": float(np.mean([r[metric] for r in results])), "median": float(np.median([r[metric] for r in results]))}
        for metric in METRIC_COLORS
    }

    # create a unique run ID based on timestamp and tag, to avoid overwriting previous runs
    timestamp = datetime.now().strftime("%Y%m%d-%H%M%S") #yearmonthday-hourminutesecond
    safe_tag = re.sub(r"[^a-zA-Z0-9_-]+", "-", tag).strip("-") or "run"
    run_id = f"{timestamp}_{safe_tag}"

    fig, ax_x = plt.subplots(figsize=(6, 4.5), facecolor=CHART_SURFACE)

    # mean vs median for each metric graph
    ax_x.set_facecolor(CHART_SURFACE)
    x = np.arange(len(METRIC_COLORS))   
    width = 0.32

    # plot mean values first, median values then
    for offset, stat, hatch in [(-1, "mean", None), (1, "median", "///")]: # offset = to not overlap mean and median graphs, hatch = graph pattern
        heights = [aggregate[m][stat] for m in METRIC_COLORS] # value of each stat for the given metric
        # bar style
        bars = ax_x.bar(x + offset * width / 2, heights, width,
                           color=list(METRIC_COLORS.values()), hatch=hatch,
                           edgecolor=CHART_SURFACE, linewidth=1)
        # numbers above each bar
        for rect, h in zip(bars, heights):
            ax_x.text(rect.get_x() + rect.get_width() / 2, h + 0.02, f"{h:.2f}", ha="center", va="bottom", fontsize=8, color=CHART_INK)

    # style graph    
    ax_x.set_xticks(x)
    ax_x.set_xticklabels([m.capitalize() for m in METRIC_COLORS], color=CHART_INK)
    ax_x.set_ylim(0, 1.12) # room for the numbers above the bars
    ax_x.set_ylabel("score", color=CHART_MUTED)
    ax_x.tick_params(colors=CHART_MUTED)
    ax_x.yaxis.grid(True, color=CHART_GRID, linewidth=0.8) # add horizontal grid lines to the graph
    ax_x.set_axisbelow(True)

    # remove/modify spines (borders) of the graph
    for spine in ("top", "right"):
        ax_x.spines[spine].set_visible(False)
    for spine in ("left", "bottom"):
        ax_x.spines[spine].set_color(CHART_GRID)

    # legend
    mean_patch = plt.Rectangle((0, 0), 1, 1, facecolor=CHART_MUTED, edgecolor=CHART_SURFACE)
    median_patch = plt.Rectangle((0, 0), 1, 1, facecolor=CHART_MUTED, edgecolor=CHART_SURFACE, hatch="///")
    ax_x.legend([mean_patch, median_patch], ["mean", "median"], frameon=False, loc="upper right", fontsize=8, labelcolor=CHART_INK)
    ax_x.set_title(f"{len(results)} image{'s' if len(results) != 1 else ''} evaluated", fontsize=10, color=CHART_MUTED, loc="left")

    fig.suptitle(tag, fontsize=13, color=CHART_INK, fontweight="bold", x=0.02, y=0.99, ha="left") # title of the run, top-left
    fig.text(0.02, 0.905, timestamp, fontsize=9, color=CHART_MUTED, ha="left") # add the ID to the title

    wrapped_description = "\n".join(textwrap.wrap(description, width=140)) if description else ""
    if wrapped_description:
        fig.text(0.02, 0.04, wrapped_description, fontsize=8, color=CHART_MUTED, ha="left", va="bottom") # description

    plt.tight_layout(rect=[0, 0.12, 1, 0.88])

    summary_path = os.path.join(output_dir, f"{run_id}_summary.png") # name of the final file of the graph
    fig.savefig(summary_path, dpi=150, facecolor=CHART_SURFACE, bbox_inches="tight")
    plt.show()

    # create JSON record for the graph
    metadata = {
        "run_id": run_id,
        "tag": tag,
        "description": description,
        "timestamp": timestamp,
        "n_images": len(results),
        "aggregate": aggregate,
        "per_image": results,
    }
    with open(os.path.join(output_dir, f"{run_id}_metadata.json"), "w") as f:
        json.dump(metadata, f, indent=2)

    # add to the JSON storing the history of all runs
    history_path = os.path.join(output_dir, "history.json")
    history = []
    if os.path.exists(history_path):
        with open(history_path) as f:
            history = json.load(f)
    history.append({
        "run_id": run_id,
        "tag": tag,
        "description": description,
        "timestamp": timestamp,
        "n_images": len(results),
        "aggregate": aggregate,
    })
    with open(history_path, "w") as f:
        json.dump(history, f, indent=2)

    print(f"saved {summary_path}")
    return metadata

In [68]:
def plot_history(output_dir=GRAPHS_DIR):
    """
    Plot mean precision/recall/F1 across all logged runs.

    Args:
        output_dir: Directory containing history.json.

    Returns:
        The run history, or None if no history exists.
    """
    history_path = os.path.join(output_dir, "history.json")
    if not os.path.exists(history_path):
        print(f"no history in {history_path} yet, call log_performance() at least once")
        return

    with open(history_path) as f:
        history = json.load(f)
    if not history:
        print("history.json is empty")
        return

    tags = [run["tag"] for run in history] # get tags of the runs
    x = np.arange(len(history))

    fig, ax = plt.subplots(figsize=(max(6, len(history) * 0.9), 4.5), facecolor=CHART_SURFACE) # width proportional to #runs
    ax.set_facecolor(CHART_SURFACE)

    # plot means of each metric
    for metric, color in METRIC_COLORS.items():
        values = [run["aggregate"][metric]["mean"] for run in history]
        ax.plot(x, values, marker="o", markersize=6, linewidth=2, color=color, label=metric.capitalize())

    # style graph
    ax.set_xticks(x)
    ax.set_xticklabels(tags, rotation=45, ha="right", color=CHART_INK, fontsize=9)
    ax.set_ylim(0, 1.05)
    ax.set_ylabel("mean score", color=CHART_MUTED)
    ax.tick_params(colors=CHART_MUTED)
    ax.yaxis.grid(True, color=CHART_GRID, linewidth=0.8)
    ax.set_axisbelow(True)

    # remove/modify spines (borders) of the graph
    for spine in ("top", "right"):
        ax.spines[spine].set_visible(False)
    for spine in ("left", "bottom"):
        ax.spines[spine].set_color(CHART_GRID)

    # legend
    ax.legend(frameon=False, loc="lower right", labelcolor=CHART_INK)
    ax.set_title("performance across pipeline runs", fontsize=12, color=CHART_INK, loc="left", fontweight="bold")

    plt.tight_layout()
    trend_path = os.path.join(output_dir, "history_trend.png")
    fig.savefig(trend_path, dpi=150, facecolor=CHART_SURFACE, bbox_inches="tight")
    plt.show()

    print(f"saved {trend_path}")
    return history

## Entry points

Ready-to-run cells for the notebook's main use cases.

- **Verify a single image**: metrics for one image, every intermediate stage shown, plus the GT-vs-prediction overlay.
- **Verify a dataset**: batch precision/recall/F1 over a whole folder, optionally logged to `GRAPHS/`.
- **CAD-ready export (single image)**: thickness-aware `.svg` export, plus a footprint-over-original overlay to check it by eye.
- **CAD-ready export (dataset)**: same export, batched over a folder.

In [ ]:
if __name__ == "__main__":
    img_path = "DATASETS/CVC-FP/Ia_JN0701_sommaire.png"
    svg_path = None  # None = auto-discover f"{img_name}_gt_*.svg" next to img_path

    metrics = verify_single(img_path, svg_path, visualize=True)

In [ ]:
if __name__ == "__main__":
    dataset_dir = "DATASETS/CVC-FP"

    results = verify_dataset(dataset_dir)

    log_performance(
         results,
         tag="walls tresh = 4",
         description="isolate walls thickness threshold = 4px",
    )
    plot_history()


In [ ]:
if __name__ == "__main__":
    img_path = "DATASETS/CVC-FP/Ia_JN0701_sommaire.png"
    output_svg = "output.svg"

    img = cv2.imread(img_path)
    if img is None:
        raise FileNotFoundError(f"Could not read image: {img_path}")

    binary_img = preprocess(img)
    walls_mask = isolate_walls(binary_img).astype(np.uint8)
    raw_segments = detect_walls(walls_mask)
    regularized = regularize(raw_segments, img.shape[:2])
    thicknesses = estimate_wall_thickness(regularized, walls_mask)
    segments_to_svg_walls(regularized, thicknesses, img.shape[:2], output_svg)

    def overlay_wall_footprints(img, segments, thicknesses, alpha=0.5, color=(0, 255, 0)):
        # same footprint geometry as segments_to_svg_walls, rasterized instead of written to SVG
        mask = np.zeros(img.shape[:2], dtype=np.uint8)
        for (x1, y1, x2, y2), thickness in zip(segments, thicknesses):
            dx, dy = x2 - x1, y2 - y1
            length = np.hypot(dx, dy)
            if length == 0:
                continue
            nx, ny = -dy / length, dx / length
            half = thickness / 2
            # add thickness
            pts = np.array([
                [x1 + nx * half, y1 + ny * half], [x2 + nx * half, y2 + ny * half],
                [x2 - nx * half, y2 - ny * half], [x1 - nx * half, y1 - ny * half],
            ], dtype=np.int32).reshape((-1, 1, 2))
            cv2.fillPoly(mask, [pts], 255)

        colored = np.zeros_like(img)
        colored[mask == 255] = color
        blended = cv2.addWeighted(img, 1.0, colored, alpha, 0)
        show_images({"Original": img, "CAD footprints": colored, "Overlay": blended})
        return blended

    overlay_wall_footprints(img, regularized, thicknesses)


In [ ]:
if __name__ == "__main__":
    input_dir = "DATASETS/CVC-FP"
    output_dir = "output_svgs"
    os.makedirs(output_dir, exist_ok=True)

    import contextlib, io

    for img_path in select_images(input_dir):
        img = cv2.imread(img_path)
        binary_img = preprocess(img)
        walls_mask = isolate_walls(binary_img).astype(np.uint8)
        raw_segments = detect_walls(walls_mask)
        regularized = regularize(raw_segments, img.shape[:2])
        thicknesses = estimate_wall_thickness(regularized, walls_mask)

        out_name = os.path.splitext(os.path.basename(img_path))[0] + ".svg"
        with contextlib.redirect_stdout(io.StringIO()):
            segments_to_svg_walls(regularized, thicknesses, img.shape[:2], os.path.join(output_dir, out_name))

        print(f"wrote {out_name}")


## Results & Discussion

## Parameters tuning analysis

### Hough parameters

| Param | Baseline | Tested range |
|---|---|---|
| `rho_px` | 1 | 1-5 |
| `theta_deg` | 1 | 0.5-5 |
| `threshold_frac` | 0.01 | 0.005-0.05 |
| `minlen_frac` | 0.01 | 0.005-0.05 |
| `maxgap_frac` | 0.005 | 0.0025-0.04 |

**rho_px and maxgap_frac** behave exactly as the geometry predicts: coarser distance/gap tolerance strictly hurts, so F1 falls monotonically over the whole tested range (and both are already best at the smallest value tested).

<div style="display:flex; gap:10px;">
  <img src="GRAPHS/Hough (rho) tuning.png" width="500">
  <img src="GRAPHS/Hough (max gap) tuning.png" width="500">
</div>

**threshold_frac and minlen_frac** show a rise-then-fall hump. Looking at precision and recall separately, they move in **opposite** directions as these parameters increase: precision climbs monotonically (a stricter vote count / longer minimum length rejects more spurious short segments) while recall falls monotonically (the same strictness also throws away real, shorter wall fragments). F1 is their harmonic mean, so it necessarily peaks somewhere in the middle of that crossover.

<div style="display:flex; gap:10px;">
  <img src="GRAPHS/Hough (threshold) tuning.png" width="500">
  <img src="GRAPHS/Hough (min len) tuning.png" width="500">
</div>

**theta_deg** register an "anomaly" at 4° and 5°, the graph raises from the first to the second value, while we would expect it to be monotonically decreasing. This is likely due to the bin logic since 4° is the only value that doesn't divide exactly the main segments' angle (90°) in floorplans

<img src="GRAPHS/Hough (theta) tuning.png" width="500">

### Regularization parameters

| Param | Baseline | Tested range |
|---|---|---|
| `angle_tolerance` | 5 | 5-25 |
| `merge_distance_frac` | 0.015 | 0.005-0.05 |
| `corner_snap_frac` | 0.007 | 0.003-0.02 |

**angle_tolerance** makes almost no difference: F1 stays flat across the whole 5-25 degree range. Segments coming out of Hough are already close enough to axis-aligned that widening the snap window barely changes anything.

<img src="GRAPHS/regularization (angle tol) tuning.png" width="500">

**merge_distance_frac** peaks at 0.01 (F1 0.702) and drops off on both sides: too tight (0.005) leaves real fragments unmerged, too loose (0.02 and up) starts fusing segments from different walls together.

<img src="GRAPHS/regularization (merge dist) tuning.png" width="500">

**corner_snap_frac** best at the tightest value tested (0.003, F1 0.697), then falls off monotonically as it grows, since a looser snap radius starts dragging endpoints toward corners they don't actually belong to.

<img src="GRAPHS/regularization (corner snap dist) tuning.png" width="500">

### Evaluation tolerance

As already noted above, `tolerance_frac` dwarfs every pipeline parameter: 0.6911 -> 0.8752 -> 0.8995 -> 0.9205 across just 0.0025 -> 0.0075 -> 0.01 -> 0.02. This is the scoring criterion, not the pipeline, so it doesn't represent an accuracy improvement, but it's a reminder that every F1 numbe is only meaningful alongside the tolerance it was computed with.

<img src="GRAPHS/tolerance tuning.png" width="500">

## Final parameters choice

Based on what we saw in the parameters tuning, we can safely set the parameters to

| Param | Baseline | Final Decision |
|---|---|---|
| `rho_px` | 1 | 2 |
| `theta_deg` | 1 | 1 |
| `threshold_frac` | 0.01 | 0.02 |
| `minlen_frac` | 0.01 | 0.01 |
| `maxgap_frac` | 0.005 | 0.0025 |
| `angle_tolerance` | 5 | 5 |
| `merge_distance_frac` | 0.015 | 0.01 |
| `corner_snap_frac` | 0.007 | 0.003 |

<br>

<div style="display:flex; gap:10px;">
  <img src="GRAPHS/baseline VS final.png" width="400">
  <img src="GRAPHS/20260830-111521_baseline_summary.png" width="400">
  <img src="GRAPHS/20260830-111721_final_summary.png" width="400">
</div>

### Interpretation

1.**Recall consistently exceeds precision**, both in the dataset-wide numbers and across almost every run in the history above. This means the pipeline's dominant error **is over-detection**, not under-detection: it finds the great majority of real walls, but also draws a meaningful amount of extra, spurious geometry, most plausibly **leaking through from room-label text, dimension lines, door-swing arcs, and furniture symbols** that survive `isolate_walls`'s thickness filtering, plus genuine over-fragmentation from Hough on noisy skeletons.

2.**mean consistently below median** on every metric confirms a left-skewed score distribution: a core of floorplans score very well, while a smaller set of harder cases, denser layouts, thinner/hatched double-line wall-drawing conventions, drag the mean down. A natural next step is targeted error analysis on that low-scoring tail specifically, rather than only tracking the aggregate.

## Project

### Known limitations

1. F1 is highly sensitive to the matching tolerance; the headline dataset mean of ~0.82 should be read as "82% of wall length matched within a resolution-proportional tolerance", not as an absolute, tolerance-free correctness figure.

2. The pipeline works well on **standard** floorplan images (black thick and filled walls, thinner line for texts, forniture etc.) and has a lot of problems with more complicated images (hatched walls, not filled, a lot of texts and forniture with thicker line etc.).

### Possible future work

- Integration of `isolate_walls_multiscale`' in the pipeline for a different analysis based on the type of wall.
- Per-image error analysis on the low-F1 tail of the CVC-FP distribution to characterize which floorplan styles are hardest and why.
- Target precision specifically: stronger filtering of text/dimension-line/furniture ink before or during wall isolation (e.g. connected-component size/shape filtering to drop text-like blobs before the distance transform).
